# Aufgaben Blatt 4 KI Machine Learning I

## Aufgabe 1 (Lineare Regression)

Bearbeiten Sie die Aufgabe https://github.com/oduerr/ki/blob/main/linear_regression/lr_gradient_descent.ipynb

Versuchen Sie den Code zu verstehen und machen die kleineren Aufgaben, die in dem notebook besprochen werden.

**→ Gelöst im Notebook `../linear_regression/lr_gradient_descent.ipynb`**

**Exercise 1:** Bessere Werte für a und b gefunden: `a = 0.9, b = 101` → MSE = 375.36 (kleiner als 408.15)

**Exercise 2:** Die optimale Lernrate liegt bei ca. `eta = 0.001`. Bei zu großen Werten (z.B. 0.01) divergiert der Gradient Descent (springt über das Minimum). Bei zu kleinen Werten (z.B. 0.00001) braucht er sehr viele Schritte.

## Aufgabe 2 (Titanic)
In dieser Aufgabe nehmen Sie an der Titanic Challenge (https://www.kaggle.com/c/titanic) teil.

In [ ]:
# Imports
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings('ignore')

### a) Train/Val-Split und regelbasierte Klassifikation nach Pclass

**Idee:** Wir schauen uns an, wie die Überlebensrate nach Ticket-Klasse (Pclass) verteilt ist.
- Pclass = 1 (1. Klasse) → Reichere Passagiere, höhere Überlebenschance
- Pclass = 3 (3. Klasse) → Ärmere Passagiere, geringere Überlebenschance

**Regel:** Pclass 1 oder 2 → überlebt (1), Pclass 3 → gestorben (0)

In [ ]:
# Trainingsdaten einlesen
train_val = pd.read_csv('titanic/train.csv')

# Kreuzaufstellung: Überleben nach Pclass analysieren
print(pd.crosstab(train_val['Pclass'], train_val['Survived'], margins=True))
print()

# Überlebensrate pro Klasse berechnen
survival_rate = train_val.groupby('Pclass')['Survived'].mean()
print('Überlebensrate pro Klasse:')
print(survival_rate)

In [ ]:
# Train/Validation Split: 80% Training, 20% Validierung
# random_state=42 sorgt für Reproduzierbarkeit
train, val = train_test_split(train_val, test_size=0.2, random_state=42)
print(f'Trainingsdaten: {len(train)} Datenpunkte')
print(f'Validierungsdaten: {len(val)} Datenpunkte')

In [ ]:
# Regelbasierte Klassifikation: Pclass 1 oder 2 → überlebt, Pclass 3 → gestorben
def rule_based_prediction(df):
    return (df['Pclass'] <= 2).astype(int)  # 1 wenn Pclass 1 oder 2, sonst 0

val_pred_rule = rule_based_prediction(val)
acc_rule = accuracy_score(val['Survived'], val_pred_rule)
print(f'Accuracy der regelbasierten Klassifikation (Validierung): {acc_rule:.4f} ({acc_rule*100:.1f}%)')

### b) Regel auf Testdaten anwenden und CSV ausgeben

In [ ]:
# Testdaten einlesen
test_data = pd.read_csv('titanic/test.csv')

# Regel auf Testdaten anwenden
pred_survived = rule_based_prediction(test_data)

# CSV ausgeben
output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': pred_survived})
output.to_csv('my_submission.csv', index=False)
print('Your submission was successfully saved!')
print(output.head())

### c) Logistische Regression mit Pclass

**Vorteil gegenüber regelbasierter Klassifikation:** Die logistische Regression lernt automatisch die beste Entscheidungsgrenze aus den Daten, statt dass wir sie manuell festlegen.

Formel *(08_linear.pdf S. 21)*:
$$P(\text{Survived} = 1 \mid \text{Pclass}) = \sigma(a \cdot \text{Pclass} + b) = \frac{1}{1 + e^{-(a \cdot \text{Pclass} + b)}}$$

In [ ]:
# Feature und Zielvariable
X_train_c = train[['Pclass']]
X_val_c = val[['Pclass']]
y_train = train['Survived']
y_val = val['Survived']

# Logistische Regression trainieren
lr_model = LogisticRegression()
lr_model.fit(X_train_c, y_train)

# Accuracy auf Validierungsset berechnen
y_pred_lr = lr_model.predict(X_val_c)
acc_lr = accuracy_score(y_val, y_pred_lr)
print(f'Accuracy Logistische Regression (nur Pclass, Validierung): {acc_lr:.4f} ({acc_lr*100:.1f}%)')

print(f'\nGelernter Koeffizient a: {lr_model.coef_[0][0]:.4f}')
print(f'Gelernter Bias b: {lr_model.intercept_[0]:.4f}')
print('→ Negativer Koeffizient: höhere Pclass-Nummer = geringere Überlebenschance')

### d) Feature Engineering

#### d.i) Missing Values (fehlende Werte behandeln)

**Was passiert hier?**
- `train['Age'].median(skipna=True)` berechnet den Median des Alters im Trainingsset (fehlende Werte werden ignoriert)
- `.fillna(...)` ersetzt alle `NaN`-Werte durch diesen Median
- **Wichtig:** Wir berechnen den Median **nur auf Trainingsdaten** und wenden ihn auf Validation-/Testdaten an → verhindert Data Leakage *(08_linear.pdf S. 27–28)*

In [ ]:
print(f'Fehlende Werte in Age (Train): {train["Age"].isna().sum()}')
print(f'Fehlende Werte in Age (Val): {val["Age"].isna().sum()}')

# Median nur auf Trainingsdaten berechnen
age_median = train["Age"].median(skipna=True)
print(f'\nMedian des Alters im Trainingsset: {age_median}')

# Fehlende Werte ersetzen (inplace=True verändert den DataFrame direkt)
val["Age"].fillna(age_median, inplace=True)
train["Age"].fillna(age_median, inplace=True)

print(f'\nFehlende Werte in Age nach Imputation (Train): {train["Age"].isna().sum()}')
print(f'Fehlende Werte in Age nach Imputation (Val): {val["Age"].isna().sum()}')

#### d.ii) Kategorische Variablen mit `pd.get_dummies` codieren

**Warum get_dummies?**
- `Sex` ist eine kategorische Variable ('male'/'female') → Zahlen benötigt
- `Pclass` ist zwar schon eine Zahl, aber wird als Kategorie behandelt (keine lineare Ordnung)
- `pd.get_dummies` erstellt Dummy-/One-Hot-Variablen: eine Spalte pro Kategorie mit 0/1-Werten

In [ ]:
# Features auswählen
features_d = ['Pclass', 'Sex', 'Age']

X_train_d = train[features_d].copy()
X_val_d = val[features_d].copy()

# One-Hot-Encoding: Pclass und Sex in numerische Werte umwandeln
X_train_d = pd.get_dummies(X_train_d, columns=['Pclass', 'Sex'])
X_val_d = pd.get_dummies(X_val_d, columns=['Pclass', 'Sex'])

# Spalten angleichen (falls Validierung eine Kategorie nicht enthält)
X_val_d = X_val_d.reindex(columns=X_train_d.columns, fill_value=0)

print('Features nach get_dummies:')
print(X_train_d.head())
print(f'\nFeature-Spalten: {list(X_train_d.columns)}')

In [ ]:
# Logistische Regression mit erweiterten Features
lr_model_d = LogisticRegression(max_iter=1000)
lr_model_d.fit(X_train_d, y_train)

y_pred_d = lr_model_d.predict(X_val_d)
acc_d = accuracy_score(y_val, y_pred_d)
print(f'Accuracy Logistische Regression (Pclass + Sex + Age, Validierung): {acc_d:.4f} ({acc_d*100:.1f}%)')
print('→ Mehr Features verbessern die Accuracy!')

### e) Weitere Klassifikatoren: Random Forest

**Was ist ein Random Forest?**
- Kombination vieler Entscheidungsbäume (Ensemble-Methode)
- Jeder Baum wird auf einem zufälligen Subset der Daten trainiert
- Endvorhersage = Mehrheitsentscheid aller Bäume
- Sehr robust und oft besser als logistische Regression bei nicht-linearen Zusammenhängen

In [ ]:
# Random Forest Klassifikator
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_d, y_train)

y_pred_rf = rf.predict(X_val_d)
acc_rf = accuracy_score(y_val, y_pred_rf)
print(f'Accuracy Random Forest (Validierung): {acc_rf:.4f} ({acc_rf*100:.1f}%)')

# Vergleich
print(f'\n--- Vergleich aller Modelle ---')
print(f'Regelbasiert (Pclass ≤ 2):      {acc_rule:.4f}')
print(f'Log. Regression (nur Pclass):   {acc_lr:.4f}')
print(f'Log. Regression (Pclass+Sex+Age): {acc_d:.4f}')
print(f'Random Forest:                  {acc_rf:.4f}')

### f) [Optional] Weitere Features und Kaggle-Upload

Mögliche zusätzliche Features: `SibSp` (Geschwister/Ehepartner an Bord), `Parch` (Eltern/Kinder an Bord), `Fare` (Ticketpreis)

---
## Aufgabe 3: Titanic mit Neuronalen Netzen

**Konzept** *(07_neuronen.pdf S. 11–27)*:
- Ein neuronales Netz besteht aus mehreren Schichten (Layers)
- Jede Schicht hat Neuronen mit Gewichten und einer Aktivierungsfunktion
- Sigmoid-Aktivierung: `σ(z) = 1/(1+e^(-z))` → Output zwischen 0 und 1
- Training via Gradient Descent auf der Binary Cross-Entropy Loss *(08_linear.pdf S. 25)*

**Loss-Funktion (Binary Cross-Entropy)**:
$$\mathcal{L} = -\frac{1}{N} \sum_{i=1}^{N} \left[ y_i \log(\hat{y}_i) + (1-y_i) \log(1-\hat{y}_i) \right]$$

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import matplotlib.pyplot as plt
%matplotlib inline

# Gleiche Daten wie in 2d verwenden
X_train_nn = X_train_d.values.astype(np.float32)
X_val_nn = X_val_d.values.astype(np.float32)
y_train_nn = y_train.values.astype(np.float32)
y_val_nn = y_val.values.astype(np.float32)

print(f'Input-Dimension: {X_train_nn.shape[1]} Features')

In [ ]:
# Neuronales Netz definieren: 2 Hidden Layer + Output Layer
# batch_input_shape=(None, X_train_nn.shape[1]) da wir X Eingabe-Features haben
model = Sequential()
model.add(Dense(16, activation='sigmoid', input_shape=(X_train_nn.shape[1],)))  # Hidden Layer 1: 16 Neuronen
model.add(Dense(8, activation='sigmoid'))                                        # Hidden Layer 2: 8 Neuronen
model.add(Dense(1, activation='sigmoid'))                                        # Output: 1 Neuron (0 oder 1)

# Adam-Optimizer + Binary Cross-Entropy Loss *(08_linear.pdf S. 25)*
opt = tf.keras.optimizers.Adam(learning_rate=1e-3)
model.compile(
    loss=tf.keras.losses.BinaryCrossentropy(),
    optimizer=opt,
    metrics=['accuracy']
)

model.summary()

In [ ]:
# Training
history = model.fit(
    X_train_nn, y_train_nn,
    validation_data=(X_val_nn, y_val_nn),
    epochs=100,
    batch_size=32,
    verbose=0
)

# Loss-Kurven plotten
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Loss
ax1.plot(history.history['loss'], label='Training Loss')
ax1.plot(history.history['val_loss'], label='Validierungs-Loss')
ax1.set_title('Loss über Epochen')
ax1.set_xlabel('Epoche')
ax1.set_ylabel('Binary Cross-Entropy Loss')
ax1.legend()

# Accuracy
ax2.plot(history.history['accuracy'], label='Training Accuracy')
ax2.plot(history.history['val_accuracy'], label='Validierungs-Accuracy')
ax2.set_title('Accuracy über Epochen')
ax2.set_xlabel('Epoche')
ax2.set_ylabel('Accuracy')
ax2.legend()

plt.tight_layout()
plt.show()

# Finale Accuracy
val_loss, val_acc = model.evaluate(X_val_nn, y_val_nn, verbose=0)
print(f'\nFinale Accuracy Neuronales Netz (Validierung): {val_acc:.4f} ({val_acc*100:.1f}%)')